# End-to-End: Segmentation ➜ Bayesian Mixture-of-Experts (MoE) ➜ Diagnostics

This notebook follows the **Mixture-of-Experts** structure from Baeldung:

- **Experts**: specialized models \(f_k(x)\) (one per segment/cluster)
- **Gating network (router)**: outputs weights \(G(x)=P(k\mid x)\) (Bayes/Naive Bayes)
- **Sparse activation**: keep only **top-k** experts per example
- **Output**: weighted sum of selected expert predictions

You start from a labeled transaction dataset (already clustered), e.g.:

- `transactions_with_kmeans_segments.csv`

## What you’ll get
1. Global **train/val/test** split (stratified by `Cluster`)
2. **Per-expert** training and **per-expert diagnostics** (val/test)
3. **Bayesian gate** diagnostics (classification report + confusion matrix)
4. **MoE vs baselines** evaluation (global model, hard routing upper-bound, MoE top-k)
5. **Routing diagnostics** (expert selection frequency, confidence, top-k hit-rate)


## 0) Imports & Config

In [17]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix, root_mean_squared_error,
)

SEED = 42
np.random.seed(SEED)

DATA_PATH = Path("transactions_with_kmeans_segments.csv")  # change if needed

# Task: "regression" (recommended) predicts log_Order_Value
#       "classification" predicts High_Value (p90)
TASK = "regression"

# Features available at prediction time
CANDIDATE_FEATURES = ["log_Quantity", "log_UnitPrice", "Country_encoded"]

# MoE sparse routing: activate only top-k experts per example
TOP_K = 2

# Global split (stratified by Cluster)
TEST_SIZE = 0.15
VAL_SIZE = 0.15  # fraction of full dataset (not of train)



## 1) Load labeled data

In [2]:
df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
df.head()


Loaded: transactions_with_kmeans_segments.csv
Shape: (530104, 17)


,Unnamed: 0,Quantity,UnitPrice,CustomerID,Country_encoded,log_Quantity,log_UnitPrice,KMeans_Cluster,DBSCAN_Cluster,HDBSCAN_Cluster,Spectral_Cluster,GMM_Cluster,PCA1,PCA2,Cluster,Country_Name,Cluster_Name
0,0,6.0,2.55,17850.0,36.0,1.945910,1.266948,3,0,9,0,0,-0.829187,-1.079836,3,United Kingdom,Low-Engagement Budget Shoppers
1,1,6.0,3.39,17850.0,36.0,1.945910,1.479329,3,0,9,0,0,-0.963173,-0.874299,3,United Kingdom,Low-Engagement Budget Shoppers
2,2,8.0,2.75,17850.0,36.0,2.197225,1.321756,2,0,9,0,0,-0.715982,-1.124107,2,United Kingdom,High-Volume Bargain Buyers
3,3,6.0,3.39,17850.0,36.0,1.945910,1.479329,3,0,9,0,0,-0.963173,-0.874299,3,United Kingdom,Low-Engagement Budget Shoppers
4,4,6.0,3.39,17850.0,36.0,1.945910,1.479329,3,0,9,0,0,-0.963173,-0.874299,3,United Kingdom,Low-Engagement Budget Shoppers


## 2) Schema checks + targets

In [3]:
required = ["CustomerID", "Cluster", "Quantity", "UnitPrice"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Ensure cluster is integer
df["Cluster"] = df["Cluster"].astype(int)

# Choose features that exist
feature_cols = [c for c in CANDIDATE_FEATURES if c in df.columns]
if not feature_cols:
    raise ValueError(f"No candidate feature columns found. Available columns: {df.columns.tolist()}")

print("Using feature columns:", feature_cols)

# Targets
df["Order_Value"] = (df["Quantity"].astype(float) * df["UnitPrice"].astype(float)).clip(lower=0)
df["log_Order_Value"] = np.log1p(df["Order_Value"])

# p90 = df["Order_Value"].quantile(0.90)
# df["High_Value"] = (df["Order_Value"] >= p90).astype(int)

# print("p90 threshold for High_Value:", p90)
# df[required + feature_cols + ["Order_Value", "log_Order_Value", "High_Value"]].head()


Using feature columns: ['log_Quantity', 'log_UnitPrice', 'Country_encoded']


## 3) Global train/val/test split (stratified by `Cluster`)

We split **rows (transactions)**, stratified by `Cluster`, so each split keeps the same segment mix.
This is the cleanest approach when your modeling unit is the **transaction** (and `Cluster` is transaction-level).


In [5]:
# Drop rows with missing required values for splitting/training
df_model = df.dropna(subset=feature_cols + ["Cluster", "log_Order_Value", ]).copy() #"High_Value"

# First split out test
trainval_df, test_df = train_test_split(
    df_model,
    test_size=TEST_SIZE,
    stratify=df_model["Cluster"],
    random_state=SEED
)

# Then split train vs val from remaining, using VAL_SIZE relative to full dataset
val_relative = VAL_SIZE / (1.0 - TEST_SIZE)

train_df, val_df = train_test_split(
    trainval_df,
    test_size=val_relative,
    stratify=trainval_df["Cluster"],
    random_state=SEED
)

print("Rows (train/val/test):", len(train_df), len(val_df), len(test_df))

def show_cluster_mix(name, d):
    print(f"\n{name} cluster proportions:")
    display(d["Cluster"].value_counts(normalize=True).sort_index().to_frame("pct"))
    print(f"{name} cluster counts:")
    display(d["Cluster"].value_counts().sort_index().to_frame("count"))

show_cluster_mix("TRAIN", train_df)
show_cluster_mix("VAL", val_df)
show_cluster_mix("TEST", test_df)


Rows (train/val/test): 371072 79516 79516

TRAIN cluster proportions:


,pct
Cluster,
0,0.063710
1,0.268447
2,0.153992
3,0.313769
4,0.200082


TRAIN cluster counts:


,count
Cluster,
0,23641
1,99613
2,57142
3,116431
4,74245



VAL cluster proportions:


...

,count
Cluster,
0,5066
1,21346
2,12244
3,24950
4,15910



TEST cluster proportions:


,pct
Cluster,
0,0.063710
1,0.268449
2,0.153982
3,0.313773
4,0.200086


TEST cluster counts:


,count
Cluster,
0,5066
1,21346
2,12244
3,24950
4,15910


## 4) Feature scaling

We fit the scaler on **TRAIN only** and apply it to VAL/TEST.


In [6]:
X_train = train_df[feature_cols].astype(float).values
X_val   = val_df[feature_cols].astype(float).values
X_test  = test_df[feature_cols].astype(float).values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

y_train_cluster = train_df["Cluster"].astype(int).values
y_val_cluster   = val_df["Cluster"].astype(int).values
y_test_cluster  = test_df["Cluster"].astype(int).values

if TASK == "regression":
    y_train = train_df["log_Order_Value"].astype(float).values
    y_val   = val_df["log_Order_Value"].astype(float).values
    y_test  = test_df["log_Order_Value"].astype(float).values
elif TASK == "classification":
    y_train = train_df["High_Value"].astype(int).values
    y_val   = val_df["High_Value"].astype(int).values
    y_test  = test_df["High_Value"].astype(int).values
else:
    raise ValueError("TASK must be 'regression' or 'classification'")

n_clusters = int(df_model["Cluster"].nunique())
print("n_clusters:", n_clusters)


n_clusters: 5


## 5) Train Experts (one model per cluster)

Each expert \(f_k(x)\) is trained **only** on TRAIN rows where `Cluster == k`.

- Regression: Ridge predicting `log_Order_Value`
- Classification: Logistic Regression predicting `High_Value`


In [7]:
expert_models = {}

for k in range(n_clusters):
    idx = (y_train_cluster == k)
    Xk = X_train_s[idx]
    yk = y_train[idx]

    if len(Xk) < 50:
        print(f"⚠️ Cluster {k}: only {len(Xk)} train rows; skipping (or reduce min cluster size upstream).")
        continue

    if TASK == "regression":
        model = Ridge(alpha=1.0, random_state=SEED)
        model.fit(Xk, yk)
    else:
        model = LogisticRegression(max_iter=200, random_state=SEED)
        model.fit(Xk, yk)

    expert_models[k] = model
    print(f"Trained expert {k}: train_rows={len(Xk):,}")

print("Experts trained:", sorted(expert_models.keys()))


Trained expert 0: train_rows=23,641
Trained expert 1: train_rows=99,613
Trained expert 2: train_rows=57,142
Trained expert 3: train_rows=116,431
Trained expert 4: train_rows=74,245
Experts trained: [0, 1, 2, 3, 4]


## 6) Expert diagnostics (per cluster)

Evaluate each expert on:
- VAL rows from the same cluster
- TEST rows from the same cluster

This answers: **"Is each expert good at its specialty?"**


In [8]:


def reg_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

def clf_metrics(y_true, y_prob, thr=0.5):
    y_hat = (y_prob >= thr).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, y_hat),
    }

def expert_predict(model, X):
    if TASK == "regression":
        return model.predict(X)
    else:
        return model.predict_proba(X)[:, 1]

rows = []
for k, model in expert_models.items():
    # VAL
    idx_val = (y_val_cluster == k)
    Xv = X_val_s[idx_val]
    yv = y_val[idx_val]

    # TEST
    idx_test = (y_test_cluster == k)
    Xt = X_test_s[idx_test]
    yt = y_test[idx_test]

    row = {
        "Cluster": k,
        "Train_Rows": int((y_train_cluster == k).sum()),
        "Val_Rows": int(idx_val.sum()),
        "Test_Rows": int(idx_test.sum()),
    }

    if idx_val.sum() > 0:
        pred_v = expert_predict(model, Xv)
        if TASK == "regression":
            row |= {f"VAL_{m}": v for m, v in reg_metrics(yv, pred_v).items()}
        else:
            row |= {f"VAL_{m}": v for m, v in clf_metrics(yv, pred_v).items()}

    if idx_test.sum() > 0:
        pred_t = expert_predict(model, Xt)
        if TASK == "regression":
            row |= {f"TEST_{m}": v for m, v in reg_metrics(yt, pred_t).items()}
        else:
            row |= {f"TEST_{m}": v for m, v in clf_metrics(yt, pred_t).items()}

    rows.append(row)

expert_metrics_df = pd.DataFrame(rows).sort_values("Cluster")
expert_metrics_df


,Cluster,Train_Rows,Val_Rows,Test_Rows,VAL_MAE,VAL_RMSE,VAL_R2,TEST_MAE,TEST_RMSE,TEST_R2
0,0,23641,5066,5066,0.155806,0.211989,0.927913,0.157078,0.212148,0.928111
1,1,99613,21346,21346,0.039083,0.048233,0.995979,0.039368,0.048932,0.995916
2,2,57142,12244,12244,0.100183,0.148631,0.979051,0.099268,0.147572,0.977856
3,3,116431,24950,24950,0.050379,0.070932,0.989623,0.050403,0.070395,0.989668
4,4,74245,15910,15910,0.091878,0.122557,0.981451,0.093678,0.126387,0.980062


## 7) Train Bayesian Gate (router)

The gate outputs a probability distribution over experts:

\[
G(x) = P(Cluster=k \mid x)
\]

We use **Gaussian Naive Bayes** (your Bayes theorem component).


In [9]:
gate = GaussianNB()
gate.fit(X_train_s, y_train_cluster)

val_gate_pred = gate.predict(X_val_s)
test_gate_pred = gate.predict(X_test_s)

print("Gate accuracy (VAL):", accuracy_score(y_val_cluster, val_gate_pred))
print("Gate accuracy (TEST):", accuracy_score(y_test_cluster, test_gate_pred))


Gate accuracy (VAL): 0.7349086976206046
Gate accuracy (TEST): 0.7321922631923135


## 8) Gate diagnostics (classification report + confusion matrix)

This answers: **"Does the router send inputs to the right experts?"**


In [10]:
print("VAL classification report:")
print(classification_report(y_val_cluster, val_gate_pred))

print("VAL confusion matrix:")
display(pd.DataFrame(confusion_matrix(y_val_cluster, val_gate_pred)))

print("TEST classification report:")
print(classification_report(y_test_cluster, test_gate_pred))

print("TEST confusion matrix:")
display(pd.DataFrame(confusion_matrix(y_test_cluster, test_gate_pred)))


VAL classification report:
              precision    recall  f1-score   support

           0       0.86      1.00      0.92      5066
           1       0.93      0.81      0.87     21346
           2       0.65      0.49      0.56     12244
           3       0.71      0.96      0.82     24950
           4       0.51      0.39      0.44     15910

    accuracy                           0.73     79516
   macro avg       0.73      0.73      0.72     79516
weighted avg       0.73      0.73      0.72     79516

VAL confusion matrix:


,0,1,2,3,4
0,5066,0,0,0,0
1,128,17277,0,3652,289
2,228,19,6052,283,5662
3,20,969,0,23838,123
4,457,270,3207,5772,6204


TEST classification report:
              precision    recall  f1-score   support

           0       0.86      1.00      0.93      5066
           1       0.93      0.81      0.86     21346
           2       0.65      0.49      0.56     12244
           3       0.71      0.96      0.82     24950
           4       0.50      0.38      0.43     15910

    accuracy                           0.73     79516
   macro avg       0.73      0.73      0.72     79516
weighted avg       0.73      0.73      0.72     79516

TEST confusion matrix:


,0,1,2,3,4
0,5066,0,0,0,0
1,126,17266,0,3628,326
2,230,11,5947,272,5784
3,18,981,0,23828,123
4,430,337,3236,5793,6114


## 9) MoE prediction (Top-k routing + weighted sum)

Baeldung sparse activation:
1. Compute gate probabilities \(G(x)\)
2. Keep only **top-k** experts
3. Weighted sum of expert outputs


In [11]:
def top_k_probs_row(probs_row, k=2, debug=False):
    probs_row = np.asarray(probs_row, dtype=float)

    if debug:
        print("\n--- Bayesian Gate Raw Output ---")
        for i, p in enumerate(probs_row):
            print(f"P(expert {i} | x) = {p:.4f}")

    # If k >= number of experts, normalize all
    if k >= len(probs_row):
        s = probs_row.sum()
        out = probs_row / s if s > 0 else np.ones_like(probs_row) / len(probs_row)

        if debug:
            print("\nUsing ALL experts (k >= n_experts)")
            print("Normalized weights:", out)

        return out

    # Indices of top-k experts
    idx = np.argsort(probs_row)[::-1][:k]

    if debug:
        print(f"\nTop-{k} experts selected:", idx.tolist())

    # Zero out non-top-k
    out = np.zeros_like(probs_row)
    out[idx] = probs_row[idx]

    if debug:
        print("\nAfter zeroing non-top-k experts:")
        for i, p in enumerate(out):
            print(f"Weight before renorm (expert {i}) = {p:.4f}")

    # Renormalize
    s = out.sum()
    out = out / s if s > 0 else out

    if debug:
        print("\nRenormalized top-k weights (sum to 1):")
        for i, p in enumerate(out):
            print(f"Final weight (expert {i}) = {p:.4f}")
        print(f"Check sum = {out.sum():.4f}")

    return out


# Global baseline (single dense model)
if TASK == "regression":
    global_model = Ridge(alpha=1.0, random_state=SEED)
else:
    global_model = LogisticRegression(max_iter=200, random_state=SEED)

global_model.fit(X_train_s, y_train)

def global_predict(X):
    if TASK == "regression":
        return global_model.predict(X)
    return global_model.predict_proba(X)[:, 1]

def hard_route_predict(X, true_cluster_labels):
    # Upper bound: route to the expert of the known true cluster
    out = np.zeros(len(X), dtype=float)
    for i, k in enumerate(true_cluster_labels):
        model = expert_models.get(int(k))
        if model is None:
            out[i] = global_predict(X[i:i+1])[0]
        else:
            out[i] = expert_predict(model, X[i:i+1])[0]
    return out

def moe_predict(X, k=2, debug_rows=3):
    probs = gate.predict_proba(X)
    out = np.zeros(len(X), dtype=float)

    for i in range(len(X)):
        debug = i < debug_rows  # only print for first few rows

        if debug:
            print("\n" + "="*60)
            print(f"MoE Prediction for sample {i}")
            print("="*60)

        # Step 1: Bayesian gate + top-k routing
        w = top_k_probs_row(probs[i], k=k, debug=debug)

        pred = 0.0
        terms = []

        # Step 2: Weighted expert predictions
        for expert_id, weight in enumerate(w):
            if weight <= 0:
                continue

            model = expert_models.get(expert_id)
            if model is None:
                yk = global_predict(X[i:i+1])[0]
                source = "Global fallback"
            else:
                yk = expert_predict(model, X[i:i+1])[0]
                source = f"Expert {expert_id}"

            term = weight * yk
            pred += term
            terms.append((expert_id, weight, yk, term, source))

            if debug:
                print(f"\n{source} prediction:")
                print(f"  ŷ_{expert_id} = {yk:.4f}")
                print(f"  weight        = {weight:.4f}")
                print(f"  contribution  = weight × prediction = {term:.4f}")

        if debug:
            print("\nFinal MoE formula:")
            formula = " + ".join(
                [f"{w:.3f}×{y:.3f}" for _, w, y, _, _ in terms]
            )
            print(f"ŷ = {formula}")
            print(f"Final blended prediction = {pred:.4f}")

        out[i] = pred

    return out



## 10) Evaluate MoE vs baselines

Baselines:
- **Global**: one model for all data (dense model)
- **Hard routing (oracle)**: choose the expert using the **true cluster** (upper bound)
- **MoE (Bayes gate + top-k)**: your final model


In [15]:
def eval_reg(y_true, y_pred, debug=False, n=5):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if debug:
        print("\n--- Regression Evaluation Debug ---")
        print(f"Showing first {n} samples\n")

        for i in range(min(n, len(y_true))):
            abs_err = abs(y_true[i] - y_pred[i])
            sq_err = (y_true[i] - y_pred[i]) ** 2

            print(f"Sample {i}:")
            print(f"  y_true (actual)      = {y_true[i]:.4f}")
            print(f"  y_pred (MoE output)  = {y_pred[i]:.4f}")
            print(f"  |error|              = {abs_err:.4f}")
            print(f"  squared error        = {sq_err:.4f}")
            print()

        print("These y_pred values should MATCH the final blended")
        print("predictions printed inside moe_predict().")
        print("------------------------------------\n")

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

def eval_clf(y_true, y_prob, thr=0.5):
    y_hat = (y_prob >= thr).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, y_hat),
    }

def evaluate_split(name, X, y, cluster_labels):
    pred_global = global_predict(X)
    pred_hard = hard_route_predict(X, cluster_labels)
    pred_moe = moe_predict(X, k=TOP_K)

    if TASK == "regression":
        rows = [
            {"Split": name, "Model": "Global (single model)"} | eval_reg(y, pred_global),
            {"Split": name, "Model": "Hard routing (oracle)"} | eval_reg(y, pred_hard),
            {"Split": name, "Model": f"MoE (Bayes gate, top-{TOP_K})"} | eval_reg(y, pred_moe,debug=True),
        ]
    else:
        rows = [
            {"Split": name, "Model": "Global (single model)"} | eval_clf(y, pred_global),
            {"Split": name, "Model": "Hard routing (oracle)"} | eval_clf(y, pred_hard),
            {"Split": name, "Model": f"MoE (Bayes gate, top-{TOP_K})"} | eval_clf(y, pred_moe,),
        ]
    return pd.DataFrame(rows)

val_results = evaluate_split("VAL", X_val_s, y_val, y_val_cluster)
test_results = evaluate_split("TEST", X_test_s, y_test, y_test_cluster)

print("Validation results:")
display(val_results)

print("Test results:")
display(test_results)



MoE Prediction for sample 0

--- Bayesian Gate Raw Output ---
P(expert 0 | x) = 0.0000
P(expert 1 | x) = 0.9489
P(expert 2 | x) = 0.0000
P(expert 3 | x) = 0.0500
P(expert 4 | x) = 0.0011

Top-2 experts selected: [1, 3]

After zeroing non-top-k experts:
Weight before renorm (expert 0) = 0.0000
Weight before renorm (expert 1) = 0.9489
Weight before renorm (expert 2) = 0.0000
Weight before renorm (expert 3) = 0.0500
Weight before renorm (expert 4) = 0.0000

Renormalized top-k weights (sum to 1):
Final weight (expert 0) = 0.0000
Final weight (expert 1) = 0.9499
Final weight (expert 2) = 0.0000
Final weight (expert 3) = 0.0501
Final weight (expert 4) = 0.0000
Check sum = 1.0000

Expert 1 prediction:
  ŷ_1 = 2.0066
  weight        = 0.9499
  contribution  = weight × prediction = 1.9061

Expert 3 prediction:
  ŷ_3 = 2.2012
  weight        = 0.0501
  contribution  = weight × prediction = 0.1102

Final MoE formula:
ŷ = 0.950×2.007 + 0.050×2.201
Final blended prediction = 2.0163

MoE Predicti

,Split,Model,MAE,RMSE,R2
0,VAL,Global (single model),0.141774,0.192824,0.964193
1,VAL,Hard routing (oracle),0.070035,0.107114,0.988951
2,VAL,"MoE (Bayes gate, top-2)",0.068103,0.104736,0.989436


Test results:


,Split,Model,MAE,RMSE,R2
0,TEST,Global (single model),0.141982,0.194053,0.963303
1,TEST,Hard routing (oracle),0.070420,0.107770,0.988682
2,TEST,"MoE (Bayes gate, top-2)",0.068672,0.107084,0.988825


## 11) Routing diagnostics (explainable MoE behavior)

These help you explain to stakeholders:
- which experts get selected most often
- how confident the gate is
- how often the true cluster is in the top-k experts


In [13]:
# Gate probabilities on VAL
val_probs = gate.predict_proba(X_val_s)

# (A) Top-1 selected expert frequency
top1 = val_probs.argmax(axis=1)
top1_counts = pd.Series(top1).value_counts().sort_index().to_frame("count")
top1_counts["pct"] = top1_counts["count"] / top1_counts["count"].sum()
print("Top-1 expert selection (VAL):")
display(top1_counts)

# (B) Confidence: max probability
top1_conf = val_probs.max(axis=1)
print("Gate confidence (VAL) summary:")
display(pd.Series(top1_conf).describe().to_frame("value"))

# (C) Top-k hit rate: is true cluster among top-k?
topk_idx = np.argsort(val_probs, axis=1)[:, ::-1][:, :TOP_K]
hit = np.mean([y_val_cluster[i] in topk_idx[i] for i in range(len(y_val_cluster))])
print(f"Top-{TOP_K} hit rate on VAL (true cluster in top-k): {hit:.3f}")


Top-1 expert selection (VAL):


,count,pct
0,5899,0.074186
1,18535,0.233098
2,9259,0.116442
3,33545,0.421865
4,12278,0.154409


Gate confidence (VAL) summary:


,value
count,79516.000000
mean,0.783696
std,0.182219
min,0.336512
25%,0.598202
50%,0.818349
75%,0.955379
max,1.000000


Top-2 hit rate on VAL (true cluster in top-k): 0.958


## 12) Save artifacts (optional)

Saves:
- scaler
- gate
- global model
- expert models dict


In [14]:
import joblib

out_dir = Path("moe_artifacts")
out_dir.mkdir(exist_ok=True)

joblib.dump(scaler, out_dir / "scaler.joblib")
joblib.dump(gate, out_dir / "bayes_gate.joblib")
joblib.dump(global_model, out_dir / "global_model.joblib")
joblib.dump(expert_models, out_dir / "expert_models.joblib")

print("Saved artifacts to:", out_dir.resolve())


Saved artifacts to: /home/rapids/notebooks/moe_artifacts
